## Resources
Tabu paieškos algoritmas remiasi trimis pagrindinėmis idėjomis:
* lokalios kaimynystės paieška;
* tabu sąrašo naudojimu;
* geriausio leistino kaimyninio sprendinio pasirinkimu (su aspiracijos kriterijumi);

### Programos eiga

1. Inicializuojamas pradinis Sudoku sprendinys;
2. Sprendinys įvertinamas naudojant fitness funkciją.
3. Sugeneruojami kaimyniniai sprendiniai:
    * atliekami nedideli pakeitimai dabartiniame sprendinyje,
    * sukeičiamos dviejų nefiksuotų elementų pozicijos atsitiktinai pasirinktoje eilutėje.
4. Ieškoma geriausio kaimyninio sprendinio, kuris nėra tabu sąraše. Taip pat taikomas aspiracijos kriterijus: jei tabu judesys leidžia pasiekti geresnį globalų sprendinį, jis yra priimamas.
5. Tabu sąrašas atnaujinamas:
    * atvirkštiniai (atlikto judesio) pakeitimai laikinai uždraudžiami,
    * taip siekiama išvengti ciklų ir pakartotinio grįžimo prie ankstesnių būsenų.
6. Procesas kartojamas tol, kol randamas sprendinys (fitness=0) arba pasiekiamas iteracijų limitas.

### Algoritmo įgyvendinimo principai

* `tabu_list` – struktūra, sauganti neseniai atliktus judesius (atvirkštines operacijas).
* `generate_tabu_neighbor()` – funkcija, generuojanti vieną kaimyninį sprendinį, sukeičiant du elementus eilutėje.
* `fitness()` – apskaičiuoja Sudoku konfliktų skaičių.
* `tabu_search()` – pagrindinė tabu paieškos algoritmo funkcija, kuri koordinuoja kaimynų generavimą, geriausio kaimyno pasirinkimą (įskaitant aspiracijos kriterijų) ir tabu sąrašo atnaujinimą.

Įgyvendinimo pastaba: Sudoku uždavinyje fitness funkcija interpretuojama kaip konfliktų skaičius eilutėse, stulpeliuose ir blokuose. Mažesnė fitness reikšmė reiškia geresnį sprendinį. Tabu sąrašo dydis reguliuoja balansą tarp naujų sprendinių tyrinėjimo ir grįžimo prie ankstesnių paieškos sričių vengimo.

```latex
\documentclass{article}
\usepackage[utf8]{inputenc}
\usepackage[lithuanian]{babel}
\usepackage{enumitem}

\begin{document}

\section*{Resources}
Tabu paieškos algoritmas remiasi trimis pagrindinėmis idėjomis:
\begin{itemize}[label=\textbullet]
    \item lokalios kaimynystės paieška;
    \item tabu sąrašo naudojimu;
    \item geriausio leistino kaimyninio sprendinio pasirinkimu (su aspiracijos kriterijumi);
\end{itemize}

\subsection*{Programos eiga}

\begin{enumerate}
    \item Inicializuojamas pradinis Sudoku sprendinys;
    
    \item Sprendinys įvertinamas naudojant fitness funkciją.
    
    \item Sugeneruojami kaimyniniai sprendiniai:
    \begin{itemize}[label=\textbullet]
        \item atliekami nedideli pakeitimai dabartiniame sprendinyje,
        \item sukeičiamos dviejų nefiksuotų elementų pozicijos atsitiktinai pasirinktoje eilutėje.
    \end{itemize}
    
    \item Ieškoma geriausio kaimyninio sprendinio, kuris nėra tabu sąraše. Taip pat taikomas aspiracijos kriterijus: jei tabu judesys leidžia pasiekti geresnį globalų sprendinį, jis yra priimamas.
    
    \item Tabu sąrašas atnaujinamas:
    \begin{itemize}[label=\textbullet]
        \item atvirkštiniai (atlikto judesio) pakeitimai laikinai uždraudžiami,
        \item taip siekiama išvengti ciklų ir pakartotinio grįžimo prie ankstesnių būsenų.
    \end{itemize}
    
    \item Procesas kartojamas tol, kol randamas sprendinys (fitness=0) arba pasiekiamas iteracijų limitas.
\end{enumerate}

\subsection*{Algoritmo įgyvendinimo principai}

\begin{itemize}[label=\textbullet]
    \item \textbf{tabu\_list} – struktūra, sauganti neseniai atliktus judesius (atvirkštines operacijas).
    
    \item \textbf{generate\_tabu\_neighbor()} – funkcija, generuojanti vieną kaimyninį sprendinį, sukeičiant du elementus eilutėje.
    
    \item \textbf{fitness()} – apskaičiuoja Sudoku konfliktų skaičių.
    
    \item \textbf{tabu\_search()} – pagrindinė tabu paieškos algoritmo funkcija, kuri koordinuoja kaimynų generavimą, geriausio kaimyno pasirinkimą (įskaitant aspiracijos kriterijų) ir tabu sąrašo atnaujinimą.
\end{itemize}

Įgyvendinimo pastaba: Sudoku uždavinyje fitness funkcija interpretuojama kaip konfliktų skaičius eilutėse, stulpeliuose ir blokuose. Mažesnė fitness reikšmė reiškia geresnį sprendinį. Tabu sąrašo dydis reguliuoja balansą tarp naujų sprendinių tyrinėjimo ir grįžimo prie ankstesnių paieškos sričių vengimo.

\end{document}
```

In [ ]:
import numpy as np
import random
from copy import deepcopy
from tqdm.notebook import trange, tqdm
import pandas as pd

Let's define the Sudoku boards.

In [ ]:
sudoku_4x4 = np.array([
    [1, 2, 0, 0],
    [3, 4, 0, 0],
    [0, 0, 1, 2],
    [0, 0, 3, 4]
])

sudoku_9x9 = np.array([
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
])

Next, let's define the helper functions needed for the Tabu Search algorithm, including functions to get fixed positions, create an initial individual, and calculate fitness.

In [ ]:
def get_fixed_positions(board, N):
    fixed_pos = []
    for r_idx in range(N):
        row_fixed_pos = []
        for c_idx in range(N):
            if board[r_idx][c_idx] != 0:
                row_fixed_pos.append(c_idx)
        fixed_pos.append(row_fixed_pos)
    return fixed_pos

In [ ]:
def create_individual(board, fixed_pos, N):
    individual = deepcopy(board)
    for r_idx in range(N):
        initial_numbers = list(range(1, N + 1))
        for c_idx in fixed_pos[r_idx]:
            if board[r_idx][c_idx] in initial_numbers:
                initial_numbers.remove(board[r_idx][c_idx])
        random.shuffle(initial_numbers)
        for c_idx in range(N):
            if board[r_idx][c_idx] == 0:
                individual[r_idx][c_idx] = initial_numbers.pop(0)
    return individual

In [ ]:
def fitness(individual, N):
    violations = 0
    block_size = int(np.sqrt(N))

    # Row and Column violations
    for i in range(N):
        violations += N - len(set(individual[i, :]))  # Row violations
        violations += N - len(set(individual[:, i]))  # Column violations

    # Block violations
    for i in range(0, N, block_size):
        for j in range(0, N, block_size):
            block = individual[i:i + block_size, j:j + block_size].flatten()
            violations += N - len(set(block))

    return violations

First, let's define the parameters specific to the Tabu Search algorithm.

In [ ]:
tabu_list_size_4x4 = [5, 10]
max_iterations_ts_4x4 = 5000

tabu_list_size_9x9 = [10, 20]
max_iterations_ts_9x9 = 1000

repeats_ts = 10

In [ ]:
tabu_list_size_4x4_extended = [1, 2, 3, 5, 7, 10]
max_iterations_ts_4x4_extended = 1000

--- Running Tabu Search Experiment for 6 different Tabu List sizes on 4x4 Sudoku ---

In [ ]:
df_ts_4x4_extended = run_experiment_ts(sudoku_4x4, 4, tabu_list_size_4x4_extended, repeats_ts, max_iterations_ts_4x4_extended)
display(df_ts_4x4_extended)

TS N=4, TabuSize=1:   0%|          | 0/10 [00:00<?, ?it/s]

TS N=4, TabuSize=1:   0%|          | 0/1000 [00:00<?, ?it/s]

Solved N=4 with TabuSize=1 in 1 iterations (Repeat 1/10):
[[1 2 4 3]
 [3 4 2 1]
 [4 3 1 2]
 [2 1 3 4]]


TS N=4, TabuSize=1:   0%|          | 0/1000 [00:00<?, ?it/s]

Solved N=4 with TabuSize=1 in 2 iterations (Repeat 2/10):
[[1 2 4 3]
 [3 4 2 1]
 [4 3 1 2]
 [2 1 3 4]]


...

TS N=4, TabuSize=10:   0%|          | 0/1000 [00:00<?, ?it/s]

Solved N=4 with TabuSize=10 in 1 iterations (Repeat 9/10):
[[1 2 4 3]
 [3 4 2 1]
 [4 3 1 2]
 [2 1 3 4]]


TS N=4, TabuSize=10:   0%|          | 0/1000 [00:00<?, ?it/s]

Solved N=4 with TabuSize=10 in 2 iterations (Repeat 10/10):
[[1 2 4 3]
 [3 4 2 1]
 [4 3 1 2]
 [2 1 3 4]]


,Lentelės dydis,Tabu List Size,Sėkmės dažnis,Vid. iteracijų
0,4x4,1,10/10,1.7
1,4x4,2,10/10,2.1
2,4x4,3,10/10,1.9
3,4x4,5,10/10,1.9
4,4x4,7,10/10,1.9
5,4x4,10,10/10,2.0


In [ ]:
tabu_list_size_9x9_extended = [5, 10, 20, 30, 40, 50]
max_iterations_ts_9x9_extended = 1000

In [ ]:
def generate_tabu_neighbor(individual, fixed_pos, N):
    new_individual = deepcopy(individual)
    r_idx = random.randint(0, N - 1)
    mutable_indices = [j for j in range(N) if j not in fixed_pos[r_idx]]

    if len(mutable_indices) >= 2:
        c1, c2 = random.sample(mutable_indices, 2)
        new_individual[r_idx][c1], new_individual[r_idx][c2] = new_individual[r_idx][c2], new_individual[r_idx][c1]
        return new_individual, (r_idx, c1, new_individual[r_idx][c1], new_individual[r_idx][c2]), (r_idx, c2, new_individual[r_idx][c2], new_individual[r_idx][c1])
    return new_individual, None, None # No valid swap made

def tabu_search(board, N, iterations, tabu_list_size):
    fixed_pos = get_fixed_positions(board, N)
    current_solution = create_individual(board, fixed_pos, N)
    current_fitness = fitness(current_solution, N)

    best_solution = deepcopy(current_solution)
    best_fitness = current_fitness

    tabu_list = [] # Stores (row, col, old_value, new_value) of a moved cell

    for i in trange(iterations, desc=f"TS N={N}, TabuSize={tabu_list_size}", leave=False):
        if best_fitness == 0:
            return True, i, best_solution

        # Generate candidate neighbors
        neighbors = []
        for _ in range(N * N): # Try to generate enough distinct neighbors
            neighbor, move1, move2 = generate_tabu_neighbor(current_solution, fixed_pos, N)
            if move1 is not None and move2 is not None:
                neighbors.append((neighbor, move1, move2))

        if not neighbors:
            continue

        best_neighbor = None
        best_neighbor_fitness = float('inf')
        best_neighbor_move = None

        for neighbor_sol, move1, move2 in neighbors:
            # Aspiration criterion: if a tabu move leads to a new overall best, accept it
            neighbor_fitness = fitness(neighbor_sol, N)
            if (neighbor_fitness < best_fitness) and (
                ((move1[0], move1[1]), move1[2], move1[3]) in tabu_list or \
                ((move2[0], move2[1]), move2[2], move2[3]) in tabu_list
            ):
                best_neighbor = neighbor_sol
                best_neighbor_fitness = neighbor_fitness
                best_neighbor_move = (move1, move2)
                break # Aspiration criterion met, take this move

            # Otherwise, consider non-tabu moves or less fit tabu moves
            is_tabu_move = (((move1[0], move1[1]), move1[2], move1[3]) in tabu_list) or (((move2[0], move2[1]), move2[2], move2[3]) in tabu_list)

            if (not is_tabu_move) and (neighbor_fitness < best_neighbor_fitness):
                best_neighbor = neighbor_sol
                best_neighbor_fitness = neighbor_fitness
                best_neighbor_move = (move1, move2)

        if best_neighbor is None:
            # If all generated neighbors were tabu and didn't meet aspiration, fall back to simple random valid move
            # This is a simplification; more complex TS would use more sophisticated neighborhood generation
            candidate_neighbor = create_individual(board, fixed_pos, N)
            candidate_fitness = fitness(candidate_neighbor, N)
            if candidate_fitness < current_fitness:
                current_solution = candidate_neighbor
                current_fitness = candidate_fitness
            continue

        current_solution = best_neighbor
        current_fitness = best_neighbor_fitness

        # Update tabu list with the reverse move of what was chosen
        if best_neighbor_move is not None:
            move_to_tabu_1 = ((best_neighbor_move[0][0], best_neighbor_move[0][1]), best_neighbor_move[0][3], best_neighbor_move[0][2])
            move_to_tabu_2 = ((best_neighbor_move[1][0], best_neighbor_move[1][1]), best_neighbor_move[1][3], best_neighbor_move[1][2])
            tabu_list.append(move_to_tabu_1)
            tabu_list.append(move_to_tabu_2)

            if len(tabu_list) > tabu_list_size:
                tabu_list.pop(0)
                tabu_list.pop(0) # Pop two elements as we added two

        if current_fitness < best_fitness:
            best_fitness = current_fitness
            best_solution = deepcopy(current_solution)

    return False, iterations, best_solution

Next, we'll define a function to run the Tabu Search experiment with various parameters and collect the results.

In [ ]:
def run_experiment_ts(sudoku_board, N, tabu_list_sizes, repeats=1, max_iterations=1000):
    results = []

    for tabu_size in tabu_list_sizes:
        success_count = 0
        total_iterations = 0
        best_overall_board = None
        for r in trange(repeats, desc=f"TS N={N}, TabuSize={tabu_size}", leave=False):
            solved, iters, final_board = tabu_search(
                board=sudoku_board, N=N, iterations=max_iterations,
                tabu_list_size=tabu_size
            )
            if solved:
                success_count += 1
                print(f"Solved N={N} with TabuSize={tabu_size} in {iters} iterations (Repeat {r+1}/{repeats}):")
                print(final_board)
                if best_overall_board is None or fitness(final_board, N) < fitness(best_overall_board, N):
                    best_overall_board = final_board
            total_iterations += iters
            if not solved and (best_overall_board is None or fitness(final_board, N) < fitness(best_overall_board, N)):
                best_overall_board = final_board

        results.append({
            "Lentelės dydis": f"{N}x{N}",
            "Tabu List Size": tabu_size,
            "Sėkmės dažnis": f"{success_count}/{repeats}",
            "Vid. iteracijų": round(total_iterations / repeats, 2)
        })
        if success_count == 0 and best_overall_board is not None:
            print(f"For N={N}, TabuSize={tabu_size}: No solution found in any repeat. Best board found (fitness {fitness(best_overall_board, N)}):\n")
            print(best_overall_board)
    return pd.DataFrame(results)

In [ ]:
print("\n--- Running Tabu Search Experiments ---\n")
df_ts_4x4 = run_experiment_ts(sudoku_4x4, 4, tabu_list_size_4x4, repeats_ts, max_iterations_ts_4x4)
df_ts_9x9 = run_experiment_ts(sudoku_9x9, 9, tabu_list_size_9x9, repeats_ts, max_iterations_ts_9x9)

display(df_ts_4x4)
display(df_ts_9x9)


--- Running Tabu Search Experiments ---



TS N=4, TabuSize=5:   0%|          | 0/1 [00:00<?, ?it/s]

TS N=4, TabuSize=5:   0%|          | 0/5000 [00:00<?, ?it/s]

Solved N=4 with TabuSize=5 in 3 iterations (Repeat 1/1):
[[1 2 4 3]
 [3 4 2 1]
 [4 3 1 2]
 [2 1 3 4]]


TS N=4, TabuSize=10:   0%|          | 0/1 [00:00<?, ?it/s]

...

TS N=9, TabuSize=20:   0%|          | 0/1 [00:00<?, ?it/s]

TS N=9, TabuSize=20:   0%|          | 0/1000 [00:00<?, ?it/s]

Solved N=9 with TabuSize=20 in 652 iterations (Repeat 1/1):
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]


,Lentelės dydis,Tabu List Size,Sėkmės dažnis,Vid. iteracijų
0,4x4,5,1/1,3.0
1,4x4,10,1/1,3.0


,Lentelės dydis,Tabu List Size,Sėkmės dažnis,Vid. iteracijų
0,9x9,10,1/1,68.0
1,9x9,20,1/1,652.0


--- Running Tabu Search Experiment for 6 different Tabu List sizes on 9x9 Sudoku ---

In [ ]:
df_ts_9x9_extended = run_experiment_ts(sudoku_9x9, 9, tabu_list_size_9x9_extended, repeats_ts, max_iterations_ts_9x9_extended)
display(df_ts_9x9_extended)

TS N=9, TabuSize=5:   0%|          | 0/10 [00:00<?, ?it/s]

TS N=9, TabuSize=5:   0%|          | 0/1000 [00:00<?, ?it/s]

Solved N=9 with TabuSize=5 in 229 iterations (Repeat 1/10):
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]


TS N=9, TabuSize=5:   0%|          | 0/1000 [00:00<?, ?it/s]

Solved N=9 with TabuSize=5 in 862 iterations (Repeat 2/10):
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]


...

Solved N=9 with TabuSize=50 in 674 iterations (Repeat 7/10):
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]


TS N=9, TabuSize=50:   0%|          | 0/1000 [00:00<?, ?it/s]

TS N=9, TabuSize=50:   0%|          | 0/1000 [00:00<?, ?it/s]

TS N=9, TabuSize=50:   0%|          | 0/1000 [00:00<?, ?it/s]

,Lentelės dydis,Tabu List Size,Sėkmės dažnis,Vid. iteracijų
0,9x9,5,8/10,641.6
1,9x9,10,5/10,650.2
2,9x9,20,7/10,651.7
3,9x9,30,3/10,803.9
4,9x9,40,0/10,1000.0
5,9x9,50,2/10,896.2
